In [1]:
# 00_tree_baseline: 깊이 4 포화 이진 트리 임베딩 기준값 (ADR 0013 4항 + PLAN 7.1 tree 판정 규약)
# S1b 세션. 합성 데이터만 쓴다. 실행 환경은 C:\dev\kfm-spike (본 프로젝트 가상환경이 아니다).
# 0S 탐색 코드이므로 kfm 패키지 규칙 밖이다(PLAN 6.1a).
import os

for _name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[_name] = "1"  # numpy, torch를 불러오기 전에 고정한다

import datetime
import math
import platform
import time
import warnings

# geoopt가 쓰는 torch.jit.script에 대한 torch의 사용 중단 경고만 거른다(동작에 영향 없음)
warnings.filterwarnings("ignore", message=".*torch.jit.script.*", category=FutureWarning)

import geoopt  # noqa: E402
import numpy as np
import pandas as pd  # noqa: E402
import scipy  # noqa: E402
import torch  # noqa: E402
from scipy.optimize import minimize  # noqa: E402
from scipy.sparse import csr_matrix  # noqa: E402
from scipy.sparse.csgraph import shortest_path  # noqa: E402

torch.set_num_threads(1)
# geoopt 0.5.1의 PoincareBall은 곡률 텐서를 기본 dtype(float32)으로 만든다. 그러면 float64 좌표에서도
# 경계 근처 거리에 1e-6 규모의 상대오차가 생기므로(사전 점검 (1)이 잡는다) 기본 dtype을 float64로 올린다.
torch.set_default_dtype(torch.float64)

CFG = {
    # 데이터: 깊이 4 포화 이진 트리(노드 31개), 목표 거리 = hop 거리, 모든 쌍 i<j
    "depth": 4,
    "dim": 2,
    "seeds": tuple(range(10)),  # 시드 0-9
    "judge_seeds": tuple(range(5)),  # 판정은 시드 0-4
    "init_halfwidth": 0.1,  # 초기 좌표 U(-0.1, 0.1)^2, 두 공간 공통
    # 1단계: geoopt RiemannianAdam, float64
    "adam_lr": 0.05,
    "adam_window": 200,
    "adam_tol": 1e-6,
    "adam_max_iter": 50_000,
    "r_max": 5.0,  # 쌍곡 반지름 상한(곡률 -1). 매 단계 뒤 노름 <= tanh(r_max / 2)로 투영
    # 2단계: scipy L-BFGS-B 마무리
    "fin_ftol": 1e-12,
    "fin_maxiter": 20_000,
    "fin_maxfun": 50_000,  # ADR 미지정. scipy 기본값 15000이 maxiter 20000보다 작아 maxiter가 닿지 못한다
    # 판정 문턱 (PLAN 7.1, ADR 0013 4항. 바꾸지 않는다)
    "thr_min_ratio": 0.5,
    "thr_median_ratio": 0.8,
    "thr_euclid_ref": 0.1506,
    "thr_euclid_tol": 0.003,
}
# 사전 검증(합성, 2026-10-18)의 예상값, 시드 0-4
EXPECTED = {
    "hyp": [0.099059, 0.033939, 0.033939, 0.033939, 0.033939],
    "euc": [0.150633, 0.188722, 0.188722, 0.150633, 0.150633],
    "ratio": 0.2253,
}

print("날짜    :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", platform.python_version(), "|", platform.platform())
print("torch   :", torch.__version__, "| 스레드 =", torch.get_num_threads())
print("geoopt  :", geoopt.__version__)
print("scipy   :", scipy.__version__, "| numpy:", np.__version__, "| pandas:", pd.__version__)
for _key, _value in CFG.items():
    print(f"  {_key:<16}= {_value}")
print("  expected        =", EXPECTED)

날짜    : 2026-10-19T10:57:30
Python  : 3.11.9 | Windows-10-10.0.26200-SP0
torch   : 2.14.1+cpu | 스레드 = 1
geoopt  : 0.5.1
scipy   : 1.17.1 | numpy: 2.4.6 | pandas: 3.0.6
  depth           = 4
  dim             = 2
  seeds           = (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)
  judge_seeds     = (0, 1, 2, 3, 4)
  init_halfwidth  = 0.1
  adam_lr         = 0.05
  adam_window     = 200
  adam_tol        = 1e-06
  adam_max_iter   = 50000
  r_max           = 5.0
  fin_ftol        = 1e-12
  fin_maxiter     = 20000
  fin_maxfun      = 50000
  thr_min_ratio   = 0.5
  thr_median_ratio= 0.8
  thr_euclid_ref  = 0.1506
  thr_euclid_tol  = 0.003
  expected        = {'hyp': [0.099059, 0.033939, 0.033939, 0.033939, 0.033939], 'euc': [0.150633, 0.188722, 0.188722, 0.150633, 0.150633], 'ratio': 0.2253}


## 규약 (ADR 0013 4항 + PLAN 7.1)

- 목표: 깊이 4 포화 이진 트리(노드 31개)의 hop 거리, 모든 쌍 i<j(465쌍).
- 목적함수: 상대 stress Σ((s·p − d)/d)², 척도 s는 매 단계 닫힌 해 s* = Σ(p/d) / Σ(p/d)². 왜곡 = mean|s·p − d|/d.
- 두 공간: 2차원 푸앵카레 원판(곡률 −1, geoopt `PoincareBall`)과 2차원 유클리드. 초기값은 두 공간 공통 U(−0.1, 0.1)², 시드 0–9, float64.
- 1단계: geoopt `RiemannianAdam`(lr 0.05). 매 단계 뒤 원점 기준 쌍곡 반지름이 R_max = 5를 넘는 점을 원점 쪽으로 투영한다(노름 ≤ tanh 2.5). 유클리드는 상한이 없다. t회까지의 최소 stress L*_t가 (L*_{t−200} − L*_t)/L*_{t−200} ≤ 1e-6이면 멈추고, 최대 50,000회.
- 2단계: Adam 끝점(최소 시점이 아니라 마지막 값)에서 scipy L-BFGS-B로 마무리한다. 쌍곡은 x = tanh(r/2)·w/‖w‖, r ∈ [0, 5] 상자 제약으로 매개화한다.
- 판정은 마무리 값으로 하고, 마무리가 수렴한 시드만 쓴다. 문턱은 바꾸지 않는다.

In [2]:
def tree_hop_distances(depth):
    """깊이 depth 포화 이진 트리의 hop 거리 행렬. 노드 i(> 0)의 부모는 (i - 1) // 2."""
    n = 2 ** (depth + 1) - 1
    child = np.arange(1, n)
    adjacency = csr_matrix((np.ones(n - 1), (child, (child - 1) // 2)), shape=(n, n))
    return shortest_path(adjacency, unweighted=True, directed=False)


BALL = geoopt.PoincareBall(c=1.0)  # 곡률 -1


class Problem:
    """목표 거리 행렬 D(대칭, 대각 0)의 상삼각 쌍 i<j에 대한 상대 stress 문제."""

    def __init__(self, D):
        D = np.asarray(D, dtype=np.float64)
        self.n = len(D)
        i, j = np.triu_indices(self.n, 1)
        self.i, self.j = torch.tensor(i), torch.tensor(j)
        self.d = torch.tensor(D[i, j], dtype=torch.float64)

    def dist(self, X, space):
        if space == "poincare":
            return BALL.dist(X[self.i], X[self.j])
        return (X[self.i] - X[self.j]).norm(dim=-1)

    def stress(self, X, space):
        """(상대 stress, s*). s*는 계산 그래프에서 떼어 낸다(포락선 정리로 기울기는 정확하다)."""
        p = self.dist(X, space)
        q = p / self.d
        s = (q.sum() / (q * q).sum()).detach()
        return (((s * p - self.d) / self.d) ** 2).sum(), s

    def distortion(self, X, space):
        """왜곡 mean|s*·p − d|/d (DESIGN 6.6). X는 numpy 또는 torch."""
        X = torch.as_tensor(X, dtype=torch.float64)
        with torch.no_grad():
            _, s = self.stress(X, space)
            return float(((s * self.dist(X, space) - self.d).abs() / self.d).mean())


def np_distortion(D, X, space):
    """torch, geoopt와 별개인 numpy 구현(교차 점검용). 원판 거리는 arccosh 공식."""
    i, j = np.triu_indices(len(D), 1)
    diff2 = ((X[i] - X[j]) ** 2).sum(1)
    if space == "poincare":
        alpha = 1.0 - (X**2).sum(1)
        p = np.arccosh(1.0 + 2.0 * diff2 / (alpha[i] * alpha[j]))
    else:
        p = np.sqrt(diff2)
    d = D[i, j]
    q = p / d
    s = q.sum() / (q * q).sum()
    return float(np.mean(np.abs(s * p - d) / d))


def hyp_radius(X):
    """원점 기준 쌍곡 반지름 2 artanh(|x|) (곡률 -1)."""
    return 2.0 * np.arctanh(np.linalg.norm(X, axis=1))

In [3]:
def run_adam(prob, space, seed, cfg=CFG):
    """1단계: geoopt RiemannianAdam. 정지 시점의 좌표(마지막 값)를 돌려준다."""
    t0 = time.perf_counter()
    h = cfg["init_halfwidth"]
    init = np.random.default_rng(seed).uniform(-h, h, size=(prob.n, cfg["dim"]))
    X0 = torch.tensor(init, dtype=torch.float64)
    if space == "poincare":
        X = geoopt.ManifoldParameter(X0, manifold=BALL)
    else:
        X = torch.nn.Parameter(X0)
    opt = geoopt.optim.RiemannianAdam([X], lr=cfg["adam_lr"])
    cap = math.tanh(cfg["r_max"] / 2) if space == "poincare" else None
    window, tol = cfg["adam_window"], cfg["adam_tol"]
    best, hist, t = math.inf, [], 0
    while True:
        opt.zero_grad()
        loss, _ = prob.stress(X, space)
        best = min(best, loss.item())
        hist.append(best)  # hist[t] = L*_t, t회 단계 뒤까지의 최소 stress
        if t >= window and hist[t - window] - best <= tol * hist[t - window]:
            converged = True
            break
        if t >= cfg["adam_max_iter"]:
            converged = False
            break
        loss.backward()
        opt.step()
        t += 1
        if cap is not None:  # 매 단계 뒤 반지름 상한 투영
            with torch.no_grad():
                norm = X.norm(dim=-1, keepdim=True)
                X.copy_(torch.where(norm > cap, X * (cap / norm), X))
    return {
        "X": X.detach().numpy().copy(),
        "n_iter": t,
        "converged": converged,
        "seconds": time.perf_counter() - t0,
    }


def make_objective(prob, space):
    """L-BFGS-B용 (unpack, fun). 쌍곡 z = [r (n개), w (n x dim)]에서 x = tanh(r/2)·w/‖w‖.
    유클리드 z = 좌표를 편 것. fun(z)는 (stress, 기울기)를 돌려준다(기울기는 torch 자동미분)."""
    n = prob.n

    def unpack(z):
        if space == "poincare":
            r, w = z[:n], z[n:].reshape(n, -1)
            return torch.tanh(r / 2).unsqueeze(1) * w / w.norm(dim=-1, keepdim=True)
        return z.reshape(n, -1)

    def fun(z):
        zt = torch.tensor(z, dtype=torch.float64, requires_grad=True)
        loss, _ = prob.stress(unpack(zt), space)
        loss.backward()
        return loss.item(), zt.grad.numpy().copy()

    return unpack, fun


def finish(prob, space, X_end, options=None, cfg=CFG):
    """2단계: Adam 끝점에서 scipy L-BFGS-B 마무리. 쌍곡은 r in [0, r_max] 상자 제약."""
    if options is None:
        options = {  # ADR 0013 4항: ftol 1e-12, 최대 20,000회. 나머지는 scipy 기본값
            "ftol": cfg["fin_ftol"],
            "maxiter": cfg["fin_maxiter"],
            "maxfun": cfg["fin_maxfun"],
        }
    n, r_max = prob.n, cfg["r_max"]
    unpack, fun = make_objective(prob, space)
    if space == "poincare":
        norm = np.linalg.norm(X_end, axis=1)
        r0 = np.minimum(2.0 * np.arctanh(norm), r_max)
        w0 = np.where(
            norm[:, None] > 1e-12,
            X_end / np.maximum(norm, 1e-12)[:, None],
            np.eye(X_end.shape[1])[0],
        )
        z0 = np.concatenate([r0, w0.ravel()])
        bounds = [(0.0, r_max)] * n + [(None, None)] * w0.size
    else:
        z0, bounds = X_end.ravel().copy(), None
    t0 = time.perf_counter()
    res = minimize(fun, z0, jac=True, method="L-BFGS-B", bounds=bounds, options=options)
    X_fin = unpack(torch.tensor(res.x, dtype=torch.float64)).numpy().copy()
    out = {
        "X": X_fin,
        "distortion": prob.distortion(X_fin, space),
        "loss": float(res.fun),
        "converged": bool(res.success),  # L-BFGS-B status 0: ftol 또는 gtol 조건으로 끝남
        "message": str(res.message),
        "n_iter": int(res.nit),
        "seconds": time.perf_counter() - t0,
    }
    if space == "poincare":
        r = res.x[:n]
        out["max_radius"] = float(r.max())
        out["n_at_cap"] = int((r >= r_max - 1e-9).sum())  # r 변수가 상한(상자 경계)에 있는 점
    return out

## 사전 점검 (합성 사례)

한 직선 위의 점은 두 공간 모두 왜곡 0, 마무리 목적함수의 기울기(자동미분, s* 분리)는 유한차분과 일치해야 한다.

In [4]:
assert BALL.k.dtype == torch.float64 and float(BALL.k) == -1.0  # 곡률 -1, float64
D = tree_hop_distances(CFG["depth"])
prob = Problem(D)
iu = np.triu_indices(len(D), 1)
assert D.shape == (31, 31) and prob.d.numel() == 465
assert np.allclose(D, D.T) and (np.diag(D) == 0).all() and D[iu].min() == 1 and D.max() == 8

# (1) 직선 위의 점: 유클리드 좌표 = 거리, 원판은 원점을 지나는 측지선 위(x = tanh(h/2))
line = np.array([0.0, 1.0, 3.0, 6.0, 10.0])
D_line = np.abs(line[:, None] - line[None])
X_euc = np.stack([line, np.zeros_like(line)], axis=1)
assert Problem(D_line).distortion(X_euc, "euclid") < 1e-12
h = 0.5 * line
D_h = np.abs(h[:, None] - h[None])
X_hyp = np.stack([np.tanh(h / 2), np.zeros_like(h)], axis=1)
assert Problem(D_h).distortion(X_hyp, "poincare") < 1e-12
assert np.allclose(hyp_radius(X_hyp), h)
assert abs(np_distortion(D_h, X_hyp, "poincare")) < 1e-9

# (2) 마무리 목적함수의 기울기: 중심 유한차분과 비교
rng = np.random.default_rng(123)
for space in ("euclid", "poincare"):
    _, fun = make_objective(prob, space)
    n = prob.n
    z = np.concatenate([rng.uniform(0.2, 4.5, n), rng.normal(size=2 * n)])
    z = z if space == "poincare" else z[: 2 * n]
    _, g = fun(z)
    step = 1e-6
    g_fd = np.array(
        [(fun(z + e)[0] - fun(z - e)[0]) / (2 * step) for e in step * np.eye(len(z))]
    )
    err = np.abs(g - g_fd).max() / max(1.0, np.abs(g).max())
    print(f"{space:<9} 기울기 상대오차(유한차분) = {err:.2e}")
    assert err < 1e-6
print("사전 점검 통과: 트리 31노드·465쌍, 직선 사례 왜곡 0, 기울기 일치")

euclid    기울기 상대오차(유한차분) = 1.30e-08
poincare  기울기 상대오차(유한차분) = 1.79e-08
사전 점검 통과: 트리 31노드·465쌍, 직선 사례 왜곡 0, 기울기 일치


## 실행: 시드 0–9, 두 공간 (Adam → L-BFGS-B 마무리)

In [5]:
rows, stored = [], {}
t_all = time.perf_counter()
for seed in CFG["seeds"]:
    row = {"seed": seed}
    for space, tag in (("poincare", "hyp"), ("euclid", "euc")):
        a = run_adam(prob, space, seed)
        f = finish(prob, space, a["X"])
        stored[(space, seed)] = (a["X"], f["X"])
        row.update(
            {
                f"{tag}_dist": f["distortion"],
                f"{tag}_fin_conv": f["converged"],
                f"{tag}_adam_iter": a["n_iter"],
                f"{tag}_fin_iter": f["n_iter"],
                f"{tag}_adam_dist": prob.distortion(a["X"], space),
                f"{tag}_adam_conv": a["converged"],
                f"{tag}_fin_msg": f["message"],
                f"{tag}_np_dist": np_distortion(D, f["X"], space),
                f"{tag}_sec": a["seconds"] + f["seconds"],
            }
        )
        if space == "poincare":
            row["hyp_n_cap"], row["hyp_max_r"] = f["n_at_cap"], f["max_radius"]
    rows.append(row)
    print(
        f"seed {seed}: 쌍곡 {row['hyp_dist']:.6f} (Adam {row['hyp_adam_iter']}, "
        f"마무리 {row['hyp_fin_iter']}, 상한 {row['hyp_n_cap']}점) | "
        f"유클리드 {row['euc_dist']:.6f} (Adam {row['euc_adam_iter']}, "
        f"마무리 {row['euc_fin_iter']}) | {row['hyp_sec'] + row['euc_sec']:.1f} s"
    )
total_seconds = time.perf_counter() - t_all
print(f"전체 실행 시간 {total_seconds:.1f} s")

seed 0: 쌍곡 0.099059 (Adam 9405, 마무리 710, 상한 8점) | 유클리드 0.150633 (Adam 361, 마무리 0) | 8.0 s


seed 1: 쌍곡 0.033938 (Adam 10343, 마무리 312, 상한 4점) | 유클리드 0.188722 (Adam 374, 마무리 0) | 7.3 s


seed 2: 쌍곡 0.033939 (Adam 12098, 마무리 668, 상한 4점) | 유클리드 0.188722 (Adam 508, 마무리 0) | 8.8 s


seed 3: 쌍곡 0.033938 (Adam 11853, 마무리 489, 상한 4점) | 유클리드 0.150633 (Adam 410, 마무리 0) | 8.6 s


seed 4: 쌍곡 0.033939 (Adam 10652, 마무리 500, 상한 4점) | 유클리드 0.150633 (Adam 380, 마무리 0) | 7.8 s


seed 5: 쌍곡 0.033939 (Adam 11867, 마무리 451, 상한 4점) | 유클리드 0.150633 (Adam 349, 마무리 0) | 8.5 s


seed 6: 쌍곡 0.033939 (Adam 10997, 마무리 263, 상한 4점) | 유클리드 0.150633 (Adam 344, 마무리 0) | 7.7 s


seed 7: 쌍곡 0.195058 (Adam 50000, 마무리 175, 상한 14점) | 유클리드 0.150633 (Adam 385, 마무리 0) | 34.5 s


seed 8: 쌍곡 0.033938 (Adam 12206, 마무리 366, 상한 4점) | 유클리드 0.150633 (Adam 360, 마무리 0) | 8.6 s


seed 9: 쌍곡 0.033938 (Adam 11801, 마무리 449, 상한 4점) | 유클리드 0.150633 (Adam 345, 마무리 0) | 8.5 s
전체 실행 시간 108.4 s


## 시드별 표

In [6]:
df = pd.DataFrame(rows).set_index("seed")
fmt = "{:.6f}".format
print("쌍곡(푸앵카레 원판, 곡률 -1, R_max = 5)")
print("  dist = 마무리 왜곡, fin_conv = 마무리 수렴 여부, adam_iter/fin_iter = Adam/마무리 반복 수,")
print("  n_cap = 상한에 닿은 점 수, max_r = 최대 쌍곡 반지름, adam_dist = Adam 끝점 왜곡,")
print("  adam_conv = Adam이 plateau 규칙으로 멈췄는가(False면 50,000회 도달)")
hyp_cols = ["dist", "fin_conv", "adam_iter", "fin_iter", "n_cap", "max_r", "adam_dist", "adam_conv"]
print(df[[f"hyp_{c}" for c in hyp_cols]].rename(columns=lambda c: c[4:]).to_string(float_format=fmt))
print("\n유클리드 2차원")
euc_cols = ["dist", "fin_conv", "adam_iter", "fin_iter", "adam_dist", "adam_conv"]
print(df[[f"euc_{c}" for c in euc_cols]].rename(columns=lambda c: c[4:]).to_string(float_format=fmt))
print("\n마무리 종료 메시지(쌍곡):", df["hyp_fin_msg"].value_counts().to_dict())
print("마무리 종료 메시지(유클리드):", df["euc_fin_msg"].value_counts().to_dict())
max_np_diff = max(
    (df["hyp_dist"] - df["hyp_np_dist"]).abs().max(), (df["euc_dist"] - df["euc_np_dist"]).abs().max()
)
print(f"torch/geoopt 왜곡과 독립 numpy 구현의 최대 차이 = {max_np_diff:.2e}")
assert max_np_diff < 1e-9

쌍곡(푸앵카레 원판, 곡률 -1, R_max = 5)
  dist = 마무리 왜곡, fin_conv = 마무리 수렴 여부, adam_iter/fin_iter = Adam/마무리 반복 수,
  n_cap = 상한에 닿은 점 수, max_r = 최대 쌍곡 반지름, adam_dist = Adam 끝점 왜곡,
  adam_conv = Adam이 plateau 규칙으로 멈췄는가(False면 50,000회 도달)
         dist  fin_conv  adam_iter  fin_iter  n_cap    max_r  adam_dist  adam_conv
seed                                                                              
0    0.099059      True       9405       710      8 5.000000   0.099038       True
1    0.033938      True      10343       312      4 5.000000   0.033949       True
2    0.033939      True      12098       668      4 5.000000   0.033961       True
3    0.033938      True      11853       489      4 5.000000   0.033947       True
4    0.033939      True      10652       500      4 5.000000   0.033947       True
5    0.033939      True      11867       451      4 5.000000   0.033972       True
6    0.033939      True      10997       263      4 5.000000   0.033938       True
7    0.195058      True   

## 판정 1–3 (시드 0–4, 마무리가 수렴한 시드만)

In [7]:
ok = lambda b: "통과" if b else "미통과"  # noqa: E731
judge = list(CFG["judge_seeds"])
sub = df.loc[judge]
hyp = sub.loc[sub["hyp_fin_conv"], "hyp_dist"]
euc = sub.loc[sub["euc_fin_conv"], "euc_dist"]
print("판정에 쓴 시드: 쌍곡", list(hyp.index), "| 유클리드", list(euc.index))
if hyp.empty or euc.empty:
    verdict = None
    print("한 공간에 마무리가 수렴한 시드가 없어 판정 불가(보류)")
else:
    hyp_min, hyp_med, euc_min, euc_med = hyp.min(), hyp.median(), euc.min(), euc.median()
    ratio_min, ratio_med = hyp_min / euc_min, hyp_med / euc_med
    c1 = hyp_min <= CFG["thr_min_ratio"] * euc_min
    c2 = hyp_med <= CFG["thr_median_ratio"] * euc_med
    c3 = abs(euc_min - CFG["thr_euclid_ref"]) <= CFG["thr_euclid_tol"]
    verdict = bool(c1 and c2 and c3)
    print(f"쌍곡   최솟값 {hyp_min:.6f}  중앙값 {hyp_med:.6f}")
    print(f"유클리드 최솟값 {euc_min:.6f}  중앙값 {euc_med:.6f}")
    print(f"비율  최솟값 기준 {ratio_min:.4f}  중앙값 기준 {ratio_med:.4f}")
    print(f"1. 최솟값 기준 원판 <= 0.5 x 유클리드: {hyp_min:.6f} <= {0.5 * euc_min:.6f} -> {ok(c1)}")
    print(f"2. 중앙값 기준 원판 <= 0.8 x 유클리드: {hyp_med:.6f} <= {0.8 * euc_med:.6f} -> {ok(c2)}")
    print(f"3. 유클리드 최솟값 = 0.1506 +- 0.003: {euc_min:.6f} (차이 {abs(euc_min - 0.1506):.6f}) -> {ok(c3)}")
    print("전체 판정:", ok(verdict))

판정에 쓴 시드: 쌍곡 [0, 1, 2, 3, 4] | 유클리드 [0, 1, 2, 3, 4]
쌍곡   최솟값 0.033938  중앙값 0.033939
유클리드 최솟값 0.150633  중앙값 0.150633
비율  최솟값 기준 0.2253  중앙값 기준 0.2253
1. 최솟값 기준 원판 <= 0.5 x 유클리드: 0.033938 <= 0.075316 -> 통과
2. 중앙값 기준 원판 <= 0.8 x 유클리드: 0.033939 <= 0.120506 -> 통과
3. 유클리드 최솟값 = 0.1506 +- 0.003: 0.150633 (차이 0.000033) -> 통과
전체 판정: 통과


## 예상값과의 비교, 시드 5–9 요약

In [8]:
d_h = sub["hyp_dist"].to_numpy() - np.array(EXPECTED["hyp"])
d_e = sub["euc_dist"].to_numpy() - np.array(EXPECTED["euc"])
print("시드 0-4 예상값과의 차이(쌍곡):  ", np.round(d_h, 7))
print("시드 0-4 예상값과의 차이(유클리드):", np.round(d_e, 7))
print(f"최대 |차이| = {max(abs(d_h).max(), abs(d_e).max()):.2e} (예상값은 소수 6자리)")
if verdict is not None:
    print(f"비율 최솟값 기준 {ratio_min:.4f} / 중앙값 기준 {ratio_med:.4f} (예상 {EXPECTED['ratio']})")

late = df.loc[list(range(5, 10))]
print("\n시드 5-9")
print(late[["hyp_dist", "hyp_fin_conv", "hyp_n_cap", "euc_dist", "euc_fin_conv"]].to_string(float_format=fmt))
for tag, name in (("hyp", "쌍곡"), ("euc", "유클리드")):
    v = late[f"{tag}_dist"]
    print(f"  {name}: 최솟값 {v.min():.6f}  중앙값 {v.median():.6f}  최댓값 {v.max():.6f}")
worse = late.index[late["hyp_dist"] > late["euc_dist"]].tolist()
print("  원판이 유클리드보다 나쁜 시드(5-9):", worse)
allv = df[["hyp_dist", "euc_dist"]]
print(f"\n시드 0-9: 쌍곡 중앙값 {allv['hyp_dist'].median():.6f}, 유클리드 중앙값 {allv['euc_dist'].median():.6f}")
print("시드 0-9에서 마무리가 수렴하지 않은 경우: 쌍곡", df.index[~df["hyp_fin_conv"]].tolist(),
      "| 유클리드", df.index[~df["euc_fin_conv"]].tolist())
print("시드 0-9에서 Adam이 50,000회에 도달한 경우: 쌍곡", df.index[~df["hyp_adam_conv"]].tolist(),
      "| 유클리드", df.index[~df["euc_adam_conv"]].tolist())

시드 0-4 예상값과의 차이(쌍곡):   [-3.e-07 -5.e-07 -5.e-07 -5.e-07 -5.e-07]
시드 0-4 예상값과의 차이(유클리드): [-2.e-07 -2.e-07 -2.e-07 -2.e-07 -2.e-07]
최대 |차이| = 5.15e-07 (예상값은 소수 6자리)
비율 최솟값 기준 0.2253 / 중앙값 기준 0.2253 (예상 0.2253)

시드 5-9
      hyp_dist  hyp_fin_conv  hyp_n_cap  euc_dist  euc_fin_conv
seed                                                           
5     0.033939          True          4  0.150633          True
6     0.033939          True          4  0.150633          True
7     0.195058          True         14  0.150633          True
8     0.033938          True          4  0.150633          True
9     0.033938          True          4  0.150633          True
  쌍곡: 최솟값 0.033938  중앙값 0.033939  최댓값 0.195058
  유클리드: 최솟값 0.150633  중앙값 0.150633  최댓값 0.150633
  원판이 유클리드보다 나쁜 시드(5-9): [7]

시드 0-9: 쌍곡 중앙값 0.033939, 유클리드 중앙값 0.150633
시드 0-9에서 마무리가 수렴하지 않은 경우: 쌍곡 [] | 유클리드 []
시드 0-9에서 Adam이 50,000회에 도달한 경우: 쌍곡 [7] | 유클리드 []


## 마무리 옵션 민감도 (같은 Adam 끝점에서 마무리만 다시)

기준 옵션(ftol 1e-12, maxiter 20000)을 바꿔도 왜곡이 달라지지 않는지 본다. ftol이 작을수록 최적값에 가까워지지만 함수값 잡음 때문에 선탐색이 실패(ABNORMAL)하는 경우가 생긴다.

In [9]:
alt = {
    "ADR 옵션 + gtol 1e-10": {"ftol": 1e-12, "gtol": 1e-10, "maxiter": 20_000, "maxfun": 50_000},
    "사전검증 옵션(ftol 1e-15, gtol 1e-11, maxcor 30)": {
        "ftol": 1e-15, "gtol": 1e-11, "maxcor": 30, "maxiter": 20_000, "maxfun": 50_000,
    },
    "ftol 1e-14, gtol 1e-12": {"ftol": 1e-14, "gtol": 1e-12, "maxiter": 20_000, "maxfun": 50_000},
}
for name, opts in alt.items():
    diffs, convs, msgs = [], [], []
    for (space, seed), (x_adam, _) in stored.items():
        f = finish(prob, space, x_adam, options=opts)
        base = df.loc[seed, "hyp_dist" if space == "poincare" else "euc_dist"]
        diffs.append(abs(f["distortion"] - base))
        convs.append(f["converged"])
        if not f["converged"]:
            msgs.append(f"{space}/시드 {seed}: {f['message'][:30]}")
    print(f"{name}: 기준 대비 왜곡 최대 차이 {max(diffs):.2e}, 마무리 수렴 {sum(convs)}/{len(convs)}", msgs)

ADR 옵션 + gtol 1e-10: 기준 대비 왜곡 최대 차이 4.85e-09, 마무리 수렴 20/20 []


사전검증 옵션(ftol 1e-15, gtol 1e-11, maxcor 30): 기준 대비 왜곡 최대 차이 2.79e-07, 마무리 수렴 16/20 ['poincare/시드 2: ABNORMAL: ', 'poincare/시드 3: ABNORMAL: ', 'poincare/시드 6: ABNORMAL: ', 'poincare/시드 7: ABNORMAL: ']


ftol 1e-14, gtol 1e-12: 기준 대비 왜곡 최대 차이 2.82e-07, 마무리 수렴 14/20 ['poincare/시드 3: ABNORMAL: ', 'poincare/시드 5: ABNORMAL: ', 'poincare/시드 6: ABNORMAL: ', 'poincare/시드 7: ABNORMAL: ', 'poincare/시드 8: ABNORMAL: ', 'poincare/시드 9: ABNORMAL: ']


## 새 기준값 요약 (ADR 0013 "열린 점"에 옮길 값)

In [10]:
best_hyp_seed = int(sub.loc[sub["hyp_fin_conv"], "hyp_dist"].idxmin())
print("전체 판정:", None if verdict is None else ok(verdict))
if verdict is not None:
    print(f"쌍곡 최솟값 {hyp_min:.6f} (시드 {best_hyp_seed})")
    print(f"유클리드 최솟값 {euc_min:.6f}")
    print(f"시드 0-4 비율: 최솟값 기준 {ratio_min:.4f}, 중앙값 기준 {ratio_med:.4f}")
n_cap_seeds = df.loc[judge, ["hyp_n_cap", "hyp_max_r"]]
print("상한에 닿은 점 수(시드 0-4):", n_cap_seeds["hyp_n_cap"].tolist(),
      "| 최대 쌍곡 반지름(시드 0-4):", n_cap_seeds["hyp_max_r"].round(6).tolist())
print("상한에 닿은 점 수(시드 0-9):", df["hyp_n_cap"].tolist())
print(f"전체 실행 시간 {total_seconds:.1f} s (셀 실행 합계, 민감도 점검 제외)")

전체 판정: 통과
쌍곡 최솟값 0.033938 (시드 1)
유클리드 최솟값 0.150633
시드 0-4 비율: 최솟값 기준 0.2253, 중앙값 기준 0.2253
상한에 닿은 점 수(시드 0-4): [8, 4, 4, 4, 4] | 최대 쌍곡 반지름(시드 0-4): [5.0, 5.0, 5.0, 5.0, 5.0]
상한에 닿은 점 수(시드 0-9): [8, 4, 4, 4, 4, 4, 4, 14, 4, 4]
전체 실행 시간 108.4 s (셀 실행 합계, 민감도 점검 제외)


## 메모 (S4에 넘길 점)

- geoopt 0.5.1의 `PoincareBall(c=1.0)`는 곡률 텐서를 기본 dtype(float32)으로 만든다. float64 좌표에서도 `dist`가 경계 근처(반지름 3–5)에서 1e-6 규모의 상대오차를 낸다. 첫 셀의 `torch.set_default_dtype(torch.float64)`로 해결했고, 사전 점검 (1)(직선 위 점의 왜곡 0)이 이를 잡는다.
- 마무리는 ADR 문구(ftol 1e-12, 최대 20,000회)에 `maxfun`만 더했고 gtol은 scipy 기본값(1e-5)이다. 이 값이면 유클리드 마무리는 Adam 끝점이 이미 gtol 안이라 0회 반복으로 수렴 처리된다. 쌍곡은 대부분 gtol 또는 ftol 조건으로 끝난다. gtol을 1e-10으로 줄여도 왜곡 차이는 5e-9 이내이고 20회 모두 수렴한다(위 민감도 점검).
- ftol을 1e-15(사전 검증 옵션)나 1e-14로 낮추면 일부 시드에서 선탐색이 실패(`ABNORMAL`)해 수렴 판정을 쓸 수 없다. ADR의 ftol 1e-12가 맞다.
- 마무리 값은 ftol이 정한 정밀도 안에서만 정해진다(왜곡 소수 6자리에서 ±5e-7). 쌍곡 최적해의 왜곡은 약 0.0339385라 6자리 반올림이 0.033938과 0.033939로 갈린다. 기준값은 소수 5자리까지만 쓴다.